# A/B Testing — From First Principles to ML Model Selection
### A complete hands-on notebook: theory, statistics, real sklearn datasets, and model comparison

---

## Table of Contents

| Part | Topic |
|------|-------|
| **Part 0** | Setup & Mental Model |
| **Part 1** | A/B Testing from Scratch — the pure statistics |
| **Part 2** | Designing a Rigorous A/B Test — sample size, power, MDE |
| **Part 3** | Running an A/B Test on Real Data — conversion rate example |
| **Part 4** | Common Pitfalls — peeking, multiple testing, Simpsons paradox |
| **Part 5** | A/B Testing for ML Models — the right framework |
| **Part 6** | Model Selection on Breast Cancer Dataset — Logistic vs SVM vs Random Forest |
| **Part 7** | Model Selection on Digits Dataset — CNN-like features vs classical models |
| **Part 8** | Advanced: Bayesian A/B Testing |
| **Part 9** | Advanced: Multi-Armed Bandit — when classic A/B is too slow |
| **Part 10** | Final Decision Framework |

---
> **Philosophy:** Every model deployment is an A/B test. Treat it that way — with pre-registered hypotheses, power analysis, and statistical rigour — and you will ship better models and fewer disasters.


## Part 0 — Setup & The Core Mental Model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import norm, beta as beta_dist, chi2_contingency, mannwhitneyu
from sklearn import datasets
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score, confusion_matrix,
                              roc_curve)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#fafafa', 'axes.facecolor': '#fafafa',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.labelsize': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'font.family': 'DejaVu Sans', 'lines.linewidth': 2.2,
})

BLUE   = '#2980b9'; RED    = '#c0392b'; GREEN  = '#27ae60'
PURPLE = '#8e44ad'; ORANGE = '#e67e22'; GREY   = '#7f8c8d'; TEAL   = '#16a085'

np.random.seed(42)
print("All libraries loaded.")
print(f"  scikit-learn datasets ready")
print(f"  scipy stats ready")
print(f"  Plotting ready")

### The Core Mental Model

```
                    WORLD
         ┌──────────────────────────┐
         │                          │
    Group A (Control)         Group B (Treatment)
    Current model / UX        New model / UX
         │                          │
    Measure metric            Measure metric
         │                          │
         └────────── Compare ───────┘
                         │
              Is the difference real
              or just random noise?
                         │
              Statistical test decides
```

**The fundamental question A/B testing answers:**
> "If we ran this experiment an infinite number of times on different random samples,
> how often would we see a difference this large by pure chance?"

That probability is the **p-value**. If it's below your threshold (alpha), you conclude the difference is real.


---
## Part 1 — A/B Testing from Scratch

Let's build intuition by simulating a complete A/B test manually, without using any library functions.


In [ ]:
# ── Simulate a conversion rate A/B test from scratch ───────────
# Scenario: e-commerce checkout page redesign
# Control (A): old page, conversion rate ~ 10%
# Treatment (B): new page, true conversion rate ~ 12.5%

np.random.seed(7)
n_per_group = 1000
true_rate_A = 0.10
true_rate_B = 0.125

# Simulate user conversions (1 = converted, 0 = didn't)
group_A = np.random.binomial(1, true_rate_A, n_per_group)
group_B = np.random.binomial(1, true_rate_B, n_per_group)

p_A = group_A.mean()
p_B = group_B.mean()
n_A, n_B = len(group_A), len(group_B)

print("=" * 50)
print("  A/B TEST RESULTS — CHECKOUT REDESIGN")
print("=" * 50)
print(f"  Group A (Control)  : {group_A.sum():4d} / {n_A} = {p_A:.4f} ({p_A*100:.2f}%)")
print(f"  Group B (Treatment): {group_B.sum():4d} / {n_B} = {p_B:.4f} ({p_B*100:.2f}%)")
print(f"  Absolute lift      : {(p_B - p_A)*100:+.2f} pp")
print(f"  Relative lift      : {(p_B/p_A - 1)*100:+.1f}%")

In [ ]:
# ── Manual z-test for two proportions ─────────────────────────
# H0: p_A = p_B  (no difference)
# H1: p_B != p_A (two-tailed)

# Pooled proportion under H0
p_pool = (group_A.sum() + group_B.sum()) / (n_A + n_B)

# Standard error under H0
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))

# Z-statistic
z_stat = (p_B - p_A) / se

# Two-tailed p-value
p_value = 2 * (1 - norm.cdf(abs(z_stat)))

# 95% confidence interval for the difference
se_diff = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)
ci_lo = (p_B - p_A) - 1.96 * se_diff
ci_hi = (p_B - p_A) + 1.96 * se_diff

print("=" * 50)
print("  TWO-PROPORTION Z-TEST (MANUAL)")
print("=" * 50)
print(f"  Pooled proportion  : {p_pool:.4f}")
print(f"  Standard error     : {se:.4f}")
print(f"  Z-statistic        : {z_stat:.4f}")
print(f"  p-value (2-tailed) : {p_value:.4f}")
print(f"  95% CI for (B-A)   : ({ci_lo*100:.2f}pp, {ci_hi*100:.2f}pp)")
print()
alpha = 0.05
if p_value < alpha:
    print(f"  REJECT H0 (p={p_value:.4f} < alpha={alpha})")
    print("  -> Statistically significant. Treatment B wins.")
else:
    print(f"  FAIL TO REJECT H0 (p={p_value:.4f} >= alpha={alpha})")
    print("  -> No significant difference detected.")

In [ ]:
# ── Visualise the null distribution and where our z falls ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: null distribution with observed z
x = np.linspace(-4.5, 4.5, 400)
y = norm.pdf(x)
z_crit = norm.ppf(0.975)

axes[0].plot(x, y, color=BLUE, linewidth=2.5, label='Null distribution')
axes[0].fill_between(x, y, where=(x <= -z_crit), alpha=0.35, color=RED, label=f'Rejection region (alpha=0.05)')
axes[0].fill_between(x, y, where=(x >= z_crit),  alpha=0.35, color=RED)
axes[0].axvline(z_stat,  color=RED, linestyle='--', linewidth=2.5, label=f'z = {z_stat:.2f}')
axes[0].axvline(-z_stat, color=RED, linestyle='--', linewidth=2.5, alpha=0.5)
axes[0].axvline(z_crit,  color=ORANGE, linestyle=':', linewidth=1.5, label=f'z_crit = +-{z_crit:.2f}')
axes[0].axvline(-z_crit, color=ORANGE, linestyle=':', linewidth=1.5)
axes[0].set_title('Z-Test: Is the difference real?')
axes[0].set_xlabel('z-score')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Annotate p-value areas
p_shade = norm.sf(abs(z_stat))
axes[0].annotate(f'p/2 = {p_shade:.4f}', xy=(z_stat+0.1, 0.01), fontsize=9, color=RED)

# Right: bar chart of conversion rates with CI
groups = ['Control (A)', 'Treatment (B)']
rates  = [p_A * 100, p_B * 100]
cis    = [1.96 * np.sqrt(p*(1-p)/n)*100 for p, n in [(p_A,n_A),(p_B,n_B)]]

bars = axes[1].bar(groups, rates, color=[BLUE, GREEN], alpha=0.7, edgecolor='white', width=0.5)
axes[1].errorbar(groups, rates, yerr=cis, fmt='none', color='black', capsize=8, linewidth=2.5)
for bar, rate in zip(bars, rates):
    axes[1].text(bar.get_x()+bar.get_width()/2, rate+0.3, f'{rate:.2f}%',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title(f'Conversion Rates +- 95% CI\np={p_value:.4f}, {"Significant" if p_value<0.05 else "Not significant"}')
axes[1].set_ylim(0, max(rates)*1.3)

plt.suptitle('Manual A/B Test — Checkout Page Redesign', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Permutation test — a non-parametric sanity check ──────────
# "If we randomly shuffled labels, how often do we get a diff >= observed?"

observed_diff = p_B - p_A
all_data = np.concatenate([group_A, group_B])
labels   = np.array(['A']*n_A + ['B']*n_B)

n_permutations = 10000
perm_diffs = np.zeros(n_permutations)

for i in range(n_permutations):
    shuffled = np.random.permutation(all_data)
    perm_diffs[i] = shuffled[n_A:].mean() - shuffled[:n_A].mean()

perm_p = np.mean(np.abs(perm_diffs) >= abs(observed_diff))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(perm_diffs, bins=60, color=GREY, alpha=0.7, density=True, edgecolor='none',
        label=f'Permutation distribution ({n_permutations:,} permutations)')
ax.axvline(observed_diff,  color=RED, linestyle='--', linewidth=2.5,
           label=f'Observed diff = {observed_diff*100:+.2f}pp')
ax.axvline(-observed_diff, color=RED, linestyle='--', linewidth=2.5, alpha=0.5)

# shade extreme tails
ax.fill_between(np.linspace(abs(observed_diff), perm_diffs.max(), 100),
                norm.pdf(np.linspace(abs(observed_diff), perm_diffs.max(), 100),
                         perm_diffs.mean(), perm_diffs.std()),
                alpha=0.3, color=RED)

ax.set_xlabel('Difference in conversion rates (B - A)')
ax.set_ylabel('Density')
ax.set_title(f'Permutation Test (p={perm_p:.4f}) vs Z-test (p={p_value:.4f})')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print(f"Parametric z-test p-value  : {p_value:.4f}")
print(f"Permutation test p-value   : {perm_p:.4f}")
print("-> Both methods agree: the difference is statistically significant")

---
## Part 2 — Designing a Rigorous A/B Test

**The most common mistake:** running the test first, then checking if you have enough data.
You MUST determine sample size BEFORE you start. Here's why and how.

### Key Design Parameters
- **alpha (alpha):** Acceptable false positive rate. Usually 0.05.
- **beta (beta):** Acceptable false negative rate. Power = 1 - beta. Usually 0.80.
- **MDE (Minimum Detectable Effect):** The smallest lift you actually care about.
- **Baseline rate:** Current conversion rate (control group).


In [ ]:
# ── Sample size calculator for proportions ─────────────────────
def sample_size_two_proportions(p1, mde, alpha=0.05, power=0.80, two_tailed=True):
    # Returns required n per group. p1=baseline, mde=min detectable effect
    p2 = p1 + mde
    p_bar = (p1 + p2) / 2

    if two_tailed:
        z_alpha = norm.ppf(1 - alpha/2)
    else:
        z_alpha = norm.ppf(1 - alpha)

    z_beta = norm.ppf(power)

    # Two-proportion formula
    n = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) +
         z_beta  * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / mde**2

    return int(np.ceil(n))

# ── Interactive table: sample size vs MDE ─────────────────────
baseline = 0.10  # 10% conversion rate
print(f"Required sample size per group (baseline={baseline*100}%, alpha=0.05, power=0.80)")
print(f"{'MDE (pp)':<12} {'MDE (rel%)':<14} {'n per group':<14} {'Total n':<12} {'Duration (10k/day)'}")
print("-" * 68)
for mde_pp in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    mde = mde_pp / 100
    n = sample_size_two_proportions(baseline, mde)
    rel = mde / baseline * 100
    days = (n * 2) / 10000
    print(f"  {mde_pp:<10.1f} {rel:<14.1f} {n:<14,} {n*2:<12,} {days:.1f} days")

In [ ]:
# ── Power curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Sample size vs MDE for different baseline rates
mde_range = np.linspace(0.005, 0.10, 100)
for base, color in [(0.05, BLUE), (0.10, GREEN), (0.20, RED), (0.40, PURPLE)]:
    ns = [sample_size_two_proportions(base, mde) for mde in mde_range]
    axes[0].plot(mde_range*100, ns, color=color, linewidth=2, label=f'Baseline={base*100:.0f}%')

axes[0].set_xlabel('Minimum Detectable Effect (percentage points)')
axes[0].set_ylabel('Required Sample Size per Group')
axes[0].set_title('Sample Size vs MDE')
axes[0].set_ylim(0, 80000)
axes[0].legend(fontsize=9)

# 2. Power vs sample size for different MDEs
n_range = np.arange(100, 10001, 100)
baseline_p = 0.10

for mde_pp, color in [(1.0, BLUE), (2.0, GREEN), (3.0, RED), (5.0, PURPLE)]:
    mde = mde_pp / 100
    p2 = baseline_p + mde
    p_bar = (baseline_p + p2) / 2
    powers = []
    for n in n_range:
        se_h0 = np.sqrt(2 * p_bar * (1-p_bar) / n)
        z_crit = norm.ppf(0.975)
        se_h1 = np.sqrt(baseline_p*(1-baseline_p)/n + p2*(1-p2)/n)
        z_beta = (mde - z_crit * se_h0) / se_h1
        powers.append(norm.cdf(z_beta))
    axes[1].plot(n_range, [p*100 for p in powers], color=color, linewidth=2,
                 label=f'MDE={mde_pp:.0f}pp')

axes[1].axhline(80, color='black', linestyle='--', linewidth=1.5, label='80% target')
axes[1].set_xlabel('Sample Size per Group')
axes[1].set_ylabel('Statistical Power (%)')
axes[1].set_title('Power vs Sample Size (baseline=10%)')
axes[1].legend(fontsize=9)

plt.suptitle('A/B Test Design: Sample Size & Power Analysis', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Runtime calculator ─────────────────────────────────────────
def ab_test_planner(baseline_rate, mde_pp, daily_traffic,
                    alpha=0.05, power=0.80, traffic_split=0.5):
    mde = mde_pp / 100
    n_per_group = sample_size_two_proportions(baseline_rate, mde, alpha, power)
    n_total = n_per_group * 2
    n_treatment_daily = daily_traffic * traffic_split
    n_control_daily   = daily_traffic * (1 - traffic_split)
    days_needed = n_per_group / min(n_treatment_daily, n_control_daily)

    print("=" * 55)
    print("  A/B TEST PLAN")
    print("=" * 55)
    print(f"  Baseline rate     : {baseline_rate*100:.1f}%")
    print(f"  Target rate (MDE) : {(baseline_rate+mde)*100:.1f}%  (+{mde_pp:.1f}pp)")
    print(f"  Relative lift     : {mde/baseline_rate*100:.1f}%")
    print(f"  alpha             : {alpha}")
    print(f"  Power (1-beta)    : {power*100:.0f}%")
    print(f"  Traffic split     : {traffic_split*100:.0f}% / {(1-traffic_split)*100:.0f}%")
    print(f"  Daily traffic     : {daily_traffic:,}")
    print("-" * 55)
    print(f"  n per group       : {n_per_group:,}")
    print(f"  Total n needed    : {n_total:,}")
    print(f"  Estimated runtime : {days_needed:.1f} days ({days_needed/7:.1f} weeks)")
    print("=" * 55)
    return days_needed

# Example: e-commerce checkout
_ = ab_test_planner(
    baseline_rate=0.10,
    mde_pp=2.0,
    daily_traffic=5000,
    alpha=0.05,
    power=0.80
)

---
## Part 3 — Running a Real A/B Test (Simulated Daily Data)

Now let's simulate a test that runs day by day — including the dangerous temptation to peek at results early.


In [ ]:
# ── Simulate daily data accumulation ──────────────────────────
np.random.seed(42)
true_p_A = 0.10
true_p_B = 0.124   # true lift = +2.4pp

daily_users = 500   # per group
n_days = 30

days        = []
cumulative_pA, cumulative_pB = [], []
p_values    = []
cum_A_conv, cum_B_conv = 0, 0
cum_A_n,    cum_B_n    = 0, 0

for day in range(1, n_days + 1):
    # Today's data
    today_A = np.random.binomial(daily_users, true_p_A, 1)[0]
    today_B = np.random.binomial(daily_users, true_p_B, 1)[0]

    cum_A_conv += today_A; cum_A_n += daily_users
    cum_B_conv += today_B; cum_B_n += daily_users

    pA = cum_A_conv / cum_A_n
    pB = cum_B_conv / cum_B_n
    p_pool = (cum_A_conv + cum_B_conv) / (cum_A_n + cum_B_n)
    se = np.sqrt(p_pool * (1-p_pool) * (1/cum_A_n + 1/cum_B_n))

    if se > 0:
        z = (pB - pA) / se
        pv = 2 * (1 - norm.cdf(abs(z)))
    else:
        pv = 1.0

    days.append(day)
    cumulative_pA.append(pA)
    cumulative_pB.append(pB)
    p_values.append(pv)

df_daily = pd.DataFrame({
    'day': days, 'rate_A': cumulative_pA,
    'rate_B': cumulative_pB, 'p_value': p_values,
    'significant': [p < 0.05 for p in p_values]
})

print(df_daily.to_string(index=False))

In [ ]:
# ── Plot: p-value over time (the peeking problem) ─────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: conversion rates over time
axes[0].plot(days, [r*100 for r in cumulative_pA], color=BLUE, linewidth=2.5,
             label=f'Control A (true={true_p_A*100}%)')
axes[0].plot(days, [r*100 for r in cumulative_pB], color=GREEN, linewidth=2.5,
             label=f'Treatment B (true={true_p_B*100}%)')
axes[0].axhline(true_p_A*100, color=BLUE, linestyle=':', alpha=0.4, linewidth=1.5)
axes[0].axhline(true_p_B*100, color=GREEN, linestyle=':', alpha=0.4, linewidth=1.5)
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_title('Cumulative Conversion Rates Over Time')
axes[0].legend(fontsize=10)

# Shade after pre-planned sample size reached
n_required = sample_size_two_proportions(true_p_A, 0.02)
day_required = n_required / daily_users
axes[0].axvline(day_required, color=RED, linestyle='--', linewidth=2,
                label=f'Pre-planned end (n={n_required:,})')
axes[0].legend(fontsize=9)

# Bottom: p-value over time
sig_days = [d for d, s in zip(days, df_daily['significant']) if s]
axes[1].plot(days, p_values, color=PURPLE, linewidth=2.5, label='p-value (cumulative)')
axes[1].axhline(0.05, color=RED, linestyle='--', linewidth=2, label='alpha = 0.05')
axes[1].axvline(day_required, color=RED, linestyle='--', linewidth=2,
                label=f'Pre-planned end = day {day_required:.0f}')

# Highlight days where p < 0.05 BEFORE planned end (false significant moments)
early_sig = [(d, p) for d, p in zip(days, p_values) if p < 0.05 and d < day_required]
if early_sig:
    for d, p in early_sig:
        axes[1].axvspan(d-0.5, d+0.5, alpha=0.2, color=RED)

axes[1].set_xlabel('Day of Experiment')
axes[1].set_ylabel('p-value')
axes[1].set_title('p-value Over Time — The Peeking Problem')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)

plt.suptitle('A/B Test: Daily Monitoring (Beware of Peeking!)', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

final_p = p_values[-1]
print(f"Final p-value (day {n_days}): {final_p:.4f}")
print(f"Pre-planned n reached at: day {day_required:.0f}")
print(f"p-value at planned end:  {p_values[int(day_required)-1]:.4f}")

---
## Part 4 — Common A/B Testing Pitfalls

### 4.1 The Peeking Problem (shown above)
### 4.2 Multiple Testing (family-wise error rate)
### 4.3 Simpson's Paradox
### 4.4 Novelty Effect


In [ ]:
# ── 4.2 Multiple testing: FWER inflation ──────────────────────
np.random.seed(0)
n_simulations = 10000
alpha = 0.05

def run_null_test(n=1000):
    # Both groups from SAME distribution (H0 is TRUE)
    A = np.random.binomial(1, 0.10, n)
    B = np.random.binomial(1, 0.10, n)
    p_pool = (A.sum() + B.sum()) / (2*n)
    se = np.sqrt(p_pool*(1-p_pool)*(2/n))
    if se == 0: return 1.0
    z = (B.mean() - A.mean()) / se
    return 2 * (1 - norm.cdf(abs(z)))

# Simulate: what if you test k metrics simultaneously?
results = {}
for k_tests in [1, 3, 5, 10, 20]:
    false_positives = 0
    for _ in range(n_simulations):
        p_vals = [run_null_test() for _ in range(k_tests)]
        if any(p < alpha for p in p_vals):
            false_positives += 1
    fwer = false_positives / n_simulations
    bonf_alpha = alpha / k_tests
    results[k_tests] = {'fwer': fwer, 'bonf_alpha': bonf_alpha}

print(f"{'Tests':<8} {'Naive FWER':<14} {'Bonferroni alpha':<18} {'Corrected FWER'}")
print("-" * 55)
for k, v in results.items():
    theoretical_fwer = 1 - (1-alpha)**k
    print(f"  {k:<6} {v['fwer']*100:<14.1f}% {v['bonf_alpha']:<18.4f} ~{alpha*100:.0f}% (corrected)")

print()
print("KEY INSIGHT: Testing 10 metrics with alpha=0.05 gives ~40% chance")
print("of at least ONE false positive! Use Bonferroni correction: alpha/k")

In [ ]:
# ── 4.3 Simpson's Paradox ─────────────────────────────────────
# Scenario: A/B test on mobile app, across iOS and Android
# Treatment looks WORSE overall, but is BETTER on each platform!

# iOS users (smaller group but high converters)
ios_A = {'conversions': 90,  'total': 100}   # 90%
ios_B = {'conversions': 180, 'total': 200}   # 90%

# Android users (larger group, lower converters)
android_A = {'conversions': 40,  'total': 200}  # 20%
android_B = {'conversions': 10,  'total': 50}   # 20%

# Rates per platform
print("── Per-Platform Rates ───────────────────────────────")
print(f"  iOS     : A={ios_A['conversions']/ios_A['total']*100:.0f}%,  B={ios_B['conversions']/ios_B['total']*100:.0f}%  -> B same as A")
print(f"  Android : A={android_A['conversions']/android_A['total']*100:.0f}%, B={android_B['conversions']/android_B['total']*100:.0f}% -> B same as A")

# Aggregate (Simpsons paradox strikes!)
total_A_conv  = ios_A['conversions']  + android_A['conversions']   # 130
total_A_total = ios_A['total']        + android_A['total']          # 300
total_B_conv  = ios_B['conversions']  + android_B['conversions']   # 190
total_B_total = ios_B['total']        + android_B['total']          # 250

rate_A_agg = total_A_conv / total_A_total
rate_B_agg = total_B_conv / total_B_total

print()
print("── Aggregate (MISLEADING) ───────────────────────────")
print(f"  Group A: {total_A_conv}/{total_A_total} = {rate_A_agg*100:.1f}%")
print(f"  Group B: {total_B_conv}/{total_B_total} = {rate_B_agg*100:.1f}%")
print(f"  -> Aggregate says A ({rate_A_agg*100:.1f}%) BEATS B ({rate_B_agg*100:.1f}%)")
print()
print("PARADOX: B looks worse in aggregate because B got MORE")
print("iOS traffic (high-converting), but we mixed the groups unequally!")
print()
print("FIX: Always segment and check for confounding variables.")
print("     Use stratified assignment or run separate tests per segment.")

In [ ]:
# ── Visualise Simpson's Paradox ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Per-segment rates
segments = ['iOS', 'Android']
rates_A_seg = [90, 20]; rates_B_seg = [90, 20]
axes[0].bar(np.arange(2)-0.2, rates_A_seg, 0.4, color=BLUE, alpha=0.75, label='Control A')
axes[0].bar(np.arange(2)+0.2, rates_B_seg, 0.4, color=GREEN, alpha=0.75, label='Treatment B')
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(segments)
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_title("Per-Segment: B = A on both platforms")
axes[0].set_ylim(0, 110)
axes[0].legend()
for x, rA, rB in zip([0,1], rates_A_seg, rates_B_seg):
    axes[0].text(x-0.2, rA+1, f'{rA}%', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(x+0.2, rB+1, f'{rB}%', ha='center', fontsize=9, fontweight='bold')

# Aggregate (misleading)
agg_labels = ['Control A
(300 users)', 'Treatment B
(250 users)']
agg_rates  = [rate_A_agg*100, rate_B_agg*100]
bars = axes[1].bar(agg_labels, agg_rates, color=[BLUE, GREEN], alpha=0.75, width=0.4)
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title("Aggregate (MISLEADING): A appears better!")
axes[1].set_ylim(0, 90)
for bar, rate in zip(bars, agg_rates):
    axes[1].text(bar.get_x()+bar.get_width()/2, rate+1, f'{rate:.1f}%',
                 ha='center', fontsize=11, fontweight='bold', color=RED)
axes[1].text(0.5, 0.85, "SIMPSON'S PARADOX!", transform=axes[1].transAxes,
             ha='center', fontsize=13, fontweight='bold', color=RED,
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.suptitle("Simpson's Paradox — Aggregate Can Mislead", fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## Part 5 — A/B Testing for ML Model Selection

When choosing between ML models, the same A/B testing principles apply — but with important differences.

### Classic A/B (product) vs ML Model A/B

| Aspect | Product A/B | ML Model A/B |
|--------|------------|--------------|
| Metric | Conversion rate, CTR | Accuracy, AUC, F1, Precision |
| Unit | User session | Prediction / sample |
| Distribution | Bernoulli (0/1) | Can be continuous |
| Variance source | User behaviour | Model uncertainty |
| "Treatment" | New UI | New model architecture |

### The Right Way to Compare ML Models

```
WRONG: Train both models, compare accuracy on test set -> no uncertainty estimate

RIGHT 1: Cross-validation -> compare distributions of fold scores
RIGHT 2: Bootstrap -> resample test set, compare score distributions
RIGHT 3: McNemar test -> compare error patterns (paired)
RIGHT 4: Hold-out test -> z-test / t-test on predictions
```


In [ ]:
# ── Framework: statistical comparison of two models ────────────
def compare_models_bootstrap(model_A, model_B, X_test, y_test,
                              metric_fn=accuracy_score,
                              n_bootstrap=1000, alpha=0.05):
    # Bootstrap CI + hypothesis test. H0: metric(A)==metric(B)
    y_pred_A = model_A.predict(X_test)
    y_pred_B = model_B.predict(X_test)

    scores_A, scores_B = [], []
    n = len(y_test)

    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        scores_A.append(metric_fn(y_test[idx], y_pred_A[idx]))
        scores_B.append(metric_fn(y_test[idx], y_pred_B[idx]))

    scores_A, scores_B = np.array(scores_A), np.array(scores_B)
    diffs = scores_B - scores_A

    # 95% CI for each model
    ci_A = (np.percentile(scores_A, 2.5), np.percentile(scores_A, 97.5))
    ci_B = (np.percentile(scores_B, 2.5), np.percentile(scores_B, 97.5))
    ci_diff = (np.percentile(diffs, 2.5), np.percentile(diffs, 97.5))

    # p-value: proportion of bootstrap diffs <= 0 (or >= 0)
    p_val = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))

    return {
        'score_A': scores_A.mean(), 'ci_A': ci_A,
        'score_B': scores_B.mean(), 'ci_B': ci_B,
        'mean_diff': diffs.mean(), 'ci_diff': ci_diff,
        'p_value': p_val,
        'significant': p_val < alpha,
        'scores_A': scores_A, 'scores_B': scores_B, 'diffs': diffs
    }

def mcnemar_test(model_A, model_B, X_test, y_test):
    # McNemar test: compares error patterns of two models on same test set.
    pred_A = model_A.predict(X_test) == y_test
    pred_B = model_B.predict(X_test) == y_test

    # Contingency: A correct & B wrong, vs A wrong & B correct
    b = np.sum(pred_A & ~pred_B)   # A right, B wrong
    c = np.sum(~pred_A & pred_B)   # A wrong, B right

    chi2 = (abs(b - c) - 1)**2 / (b + c) if (b + c) > 0 else 0
    p_val = 1 - stats.chi2.cdf(chi2, df=1)

    return {'b': b, 'c': c, 'chi2': chi2, 'p_value': p_val,
            'significant': p_val < 0.05}

def cross_val_compare(model_A, model_B, X, y, cv=10, scoring='accuracy'):
    # Compare models via k-fold CV scores. Uses paired t-test on fold scores.
    scores_A = cross_val_score(model_A, X, y, cv=cv, scoring=scoring)
    scores_B = cross_val_score(model_B, X, y, cv=cv, scoring=scoring)
    t_stat, p_val = stats.ttest_rel(scores_B, scores_A)
    return {
        'scores_A': scores_A, 'scores_B': scores_B,
        'mean_A': scores_A.mean(), 'mean_B': scores_B.mean(),
        'std_A': scores_A.std(), 'std_B': scores_B.std(),
        't_stat': t_stat, 'p_value': p_val,
        'significant': p_val < 0.05
    }

print("Model comparison framework loaded.")
print("  compare_models_bootstrap() -> bootstrap CI + p-value")
print("  mcnemar_test()             -> paired error pattern test")
print("  cross_val_compare()        -> k-fold paired t-test")

---
## Part 6 — ML Model Selection: Breast Cancer Dataset

**Dataset:** sklearn's breast_cancer — 569 samples, 30 features, binary classification (malignant/benign)
**Task:** Select the best model for production deployment using rigorous A/B testing

**Scenario:** Your team's current production model is Logistic Regression (Model A — the control).
Three challenger models are being evaluated. Which one, if any, should replace it?


In [ ]:
# ── Load and explore the dataset ──────────────────────────────
data = datasets.load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print("=" * 50)
print("  BREAST CANCER DATASET")
print("=" * 50)
print(f"  Samples    : {X.shape[0]}")
print(f"  Features   : {X.shape[1]}")
print(f"  Classes    : {class_names[0]} (0) = {(y==0).sum()}")
print(f"               {class_names[1]} (1) = {(y==1).sum()}")
print(f"  Class ratio: {(y==1).mean()*100:.1f}% benign")
print()
df_bc = pd.DataFrame(X, columns=feature_names)
df_bc['target'] = y
print("Feature summary (first 5 features):")
print(df_bc[list(feature_names[:5]) + ['target']].describe().round(3).to_string())

In [ ]:
# ── EDA: feature distributions ────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for i, feat in enumerate(feature_names[:10]):
    for cls, color, label in [(0, RED, 'Malignant'), (1, BLUE, 'Benign')]:
        axes[i].hist(df_bc[df_bc['target']==cls][feat],
                     bins=20, color=color, alpha=0.55, density=True,
                     edgecolor='none', label=label)
    axes[i].set_title(feat, fontsize=8)
    axes[i].set_xlabel('')
    if i == 0: axes[i].legend(fontsize=7)

plt.suptitle('Breast Cancer: Feature Distributions (Malignant vs Benign)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Train/test split + preprocessing ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Test class balance: {(y_test==1).mean()*100:.1f}% benign")

# ── Define models (all wrapped in pipelines with scaler) ───────
models = {
    'Logistic Regression
(CONTROL)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest
(Challenger 1)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'SVM (RBF)
(Challenger 2)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
    'Gradient Boosting
(Challenger 3)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
}

# ── Train all models ───────────────────────────────────────────
trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    f1  = f1_score(y_test, model.predict(X_test))
    print(f"  {name.replace(chr(10),' '):40s}: acc={acc:.4f}, auc={auc:.4f}, f1={f1:.4f}")

In [ ]:
# ── Cross-validation comparison (paired t-test) ────────────────
control_name = 'Logistic Regression
(CONTROL)'
control_model = trained[control_name]

print("=" * 65)
print("  CROSS-VALIDATION COMPARISON (10-fold, paired t-test)")
print("  H0: Model == Logistic Regression performance")
print("=" * 65)
print(f"  {'Model':<32} {'Mean Acc':>9} {'Std':>7} {'vs Control':>12} {'p-value':>9} {'Sig?':>6}")
print("-" * 65)

cv_results = {}
control_cv = cross_val_score(control_model, X, y, cv=10, scoring='accuracy')

for name, model in trained.items():
    if name == control_name:
        cv_scores = control_cv
        print(f"  {name.replace(chr(10),' '):<32} {cv_scores.mean():.4f}  {cv_scores.std():.4f}  {'---':>12} {'---':>9} {'(base)':>6}")
    else:
        result = cross_val_compare(control_model, model, X, y, cv=10)
        cv_scores = result['scores_B']
        diff = result['mean_B'] - result['mean_A']
        sig = 'YES' if result['significant'] else 'no'
        print(f"  {name.replace(chr(10),' '):<32} {cv_scores.mean():.4f}  {cv_scores.std():.4f}  {diff:>+12.4f} {result['p_value']:>9.4f} {sig:>6}")
        cv_results[name] = result

In [ ]:
# ── Bootstrap comparison: each challenger vs control ───────────
print("=" * 65)
print("  BOOTSTRAP COMPARISON (1000 resamples)")
print("  H0: challenger accuracy == control accuracy on test set")
print("=" * 65)

boot_results = {}
for name, model in trained.items():
    if name == control_name: continue
    result = compare_models_bootstrap(
        control_model, model, X_test, y_test,
        metric_fn=accuracy_score, n_bootstrap=1000
    )
    boot_results[name] = result
    print(f"
  Challenger: {name.replace(chr(10),' ')}")
    print(f"    Control acc : {result['score_A']:.4f}  95% CI: ({result['ci_A'][0]:.4f}, {result['ci_A'][1]:.4f})")
    print(f"    Challenger  : {result['score_B']:.4f}  95% CI: ({result['ci_B'][0]:.4f}, {result['ci_B'][1]:.4f})")
    print(f"    Difference  : {result['mean_diff']:+.4f}  95% CI: ({result['ci_diff'][0]:+.4f}, {result['ci_diff'][1]:+.4f})")
    print(f"    p-value     : {result['p_value']:.4f}  -> {'SIGNIFICANT' if result['significant'] else 'not significant'}")

In [ ]:
# ── McNemar test: error patterns ────────────────────────────
print("=" * 55)
print("  McNEMAR'S TEST: Error Pattern Comparison")
print("  (tests if models fail on DIFFERENT samples)")
print("=" * 55)

for name, model in trained.items():
    if name == control_name: continue
    mc = mcnemar_test(control_model, model, X_test, y_test)
    print(f"
  Control vs {name.replace(chr(10),' ')}")
    print(f"    Control right, Challenger wrong (b): {mc['b']}")
    print(f"    Control wrong, Challenger right (c): {mc['c']}")
    print(f"    chi2 = {mc['chi2']:.3f}, p = {mc['p_value']:.4f}")
    if mc['significant']:
        winner = name if mc['c'] > mc['b'] else control_name
        print(f"    -> Significant! Challenger makes DIFFERENT errors. {winner.split(chr(10))[0]} catches more.")
    else:
        print(f"    -> Not significant: similar error patterns")

In [ ]:
# ── Comprehensive visualisation ────────────────────────────────
fig = plt.figure(figsize=(17, 13))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

model_names_short = ['LR (Control)', 'Random Forest', 'SVM (RBF)', 'Grad. Boosting']
colors_models = [GREY, BLUE, GREEN, ORANGE]

# ── 1. CV score distributions ───────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
all_cv = {}
for name, model in trained.items():
    all_cv[name] = cross_val_score(model, X, y, cv=10, scoring='accuracy')

positions = range(len(trained))
parts = ax1.violinplot([all_cv[n] for n in trained], positions=positions,
                       showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], colors_models):
    pc.set_facecolor(color); pc.set_alpha(0.5)
ax1.set_xticks(positions)
ax1.set_xticklabels(model_names_short, fontsize=9)
ax1.set_ylabel('10-fold CV Accuracy')
ax1.set_title('Cross-Validation Score Distributions')
ax1.axhline(all_cv[control_name].mean(), color=GREY, linestyle='--',
            linewidth=1.5, alpha=0.7, label='Control mean')
ax1.legend(fontsize=8)

# ── 2. Bootstrap diff distributions ────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
for (name, result), color in zip(boot_results.items(), [BLUE, GREEN, ORANGE]):
    ax2.hist(result['diffs']*100, bins=40, color=color, alpha=0.5,
             density=True, edgecolor='none',
             label=name.split('
')[0])
ax2.axvline(0, color='black', linewidth=2, linestyle='--', label='No diff')
ax2.set_xlabel('Accuracy diff (pp)')
ax2.set_title('Bootstrap: Challenger - Control')
ax2.legend(fontsize=7)

# ── 3. ROC curves ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
for (name, model), color in zip(trained.items(), colors_models):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax3.plot(fpr, tpr, color=color, linewidth=2,
             label=f"{name.split(chr(10))[0]} (AUC={auc:.3f})")
ax3.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax3.set_xlabel('False Positive Rate')
ax3.set_ylabel('True Positive Rate')
ax3.set_title('ROC Curves — All Models')
ax3.legend(fontsize=8)

# ── 4. 95% CI comparison ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
names_short = [n.split('
')[0] for n in trained.keys()]
means = [all_cv[n].mean() for n in trained.keys()]
stds  = [all_cv[n].std()  for n in trained.keys()]
cis   = [1.96 * s / np.sqrt(10) for s in stds]
ax4.barh(names_short, means, color=colors_models, alpha=0.65, edgecolor='none')
ax4.errorbar(means, names_short, xerr=cis, fmt='none', color='black',
             capsize=5, linewidth=1.8)
ax4.set_xlabel('Mean CV Accuracy')
ax4.set_title('Point Estimates +- 95% CI')
ax4.set_xlim(min(means)*0.985, 1.002)

# ── 5. Metric radar/summary bar ─────────────────────────────────
ax5 = fig.add_subplot(gs[2, :])
metrics = ['Accuracy', 'F1', 'AUC-ROC', 'Precision', 'Recall']
metric_fns = [
    lambda y,yp: accuracy_score(y, yp),
    lambda y,yp: f1_score(y, yp),
    lambda y,yp,ypr: roc_auc_score(y, ypr),
    lambda y,yp: precision_score(y, yp),
    lambda y,yp: recall_score(y, yp),
]

x = np.arange(len(metrics))
bar_width = 0.2
for i, (name, model) in enumerate(trained.items()):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    scores = [
        accuracy_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
    ]
    bars = ax5.bar(x + i*bar_width, scores, bar_width,
                   color=colors_models[i], alpha=0.75, label=name.split('
')[0])

ax5.set_xticks(x + bar_width*1.5)
ax5.set_xticklabels(metrics)
ax5.set_ylim(0.88, 1.01)
ax5.set_ylabel('Score')
ax5.set_title('All Metrics Comparison Across Models')
ax5.legend(fontsize=9)

plt.suptitle('Breast Cancer: Comprehensive Model A/B Comparison', fontsize=15,
             fontweight='bold', y=1.01)
plt.savefig('/tmp/bc_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved figure.")

In [ ]:
# ── Final verdict: Breast Cancer ──────────────────────────────
print("=" * 60)
print("  FINAL VERDICT: BREAST CANCER MODEL SELECTION")
print("=" * 60)

control_acc = accuracy_score(y_test, trained[control_name].predict(X_test))
control_auc = roc_auc_score(y_test, trained[control_name].predict_proba(X_test)[:,1])

best_challenger = None
best_improvement = 0

for name, model in trained.items():
    if name == control_name: continue
    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    boot = boot_results[name]
    improvement = acc - control_acc
    sig = boot['significant']
    ci_lo, ci_hi = boot['ci_diff']

    decision = "DEPLOY" if (sig and improvement > 0) else "KEEP CONTROL"

    print(f"
  Challenger: {name.split(chr(10))[0]}")
    print(f"    Accuracy improvement: {improvement*100:+.2f}pp")
    print(f"    95% CI for diff:      ({ci_lo*100:+.2f}pp, {ci_hi*100:+.2f}pp)")
    print(f"    p-value:              {boot['p_value']:.4f}")
    print(f"    Statistically sig:    {sig}")
    print(f"    DECISION: {decision}")

    if sig and improvement > best_improvement:
        best_improvement = improvement
        best_challenger = name

print()
if best_challenger:
    print(f"  RECOMMENDATION: Deploy {best_challenger.split(chr(10))[0]}")
    print(f"  -> Statistically significant improvement of {best_improvement*100:+.2f}pp")
    print(f"  -> In medical context (cancer detection), also check Recall (sensitivity)")
else:
    print(f"  RECOMMENDATION: Keep current Logistic Regression")
    print(f"  -> No challenger showed statistically significant improvement")

---
## Part 7 — ML Model Selection: Digits Dataset (Multi-Class)

**Dataset:** sklearn's digits — 1797 samples, 64 features (8x8 pixel images), 10 classes (0-9)
**New wrinkle:** Multi-class classification requires different metrics and the A/B framework extends naturally.

**Scenario:** Your team currently uses a KNN classifier (control). You want to know if any of three challengers are significantly better.


In [ ]:
# ── Load digits dataset ────────────────────────────────────────
digits = datasets.load_digits()
X_dig, y_dig = digits.data, digits.target

print("=" * 45)
print("  DIGITS DATASET")
print("=" * 45)
print(f"  Samples   : {X_dig.shape[0]}")
print(f"  Features  : {X_dig.shape[1]} (8x8 pixel values)")
print(f"  Classes   : {len(np.unique(y_dig))} (digits 0-9)")
print(f"  Per class : ~{len(y_dig)//10} samples each")

# Visualise some samples
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    idx = np.where(y_dig == digit)[0][0]
    axes[0, digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0, digit].set_title(f'Label: {digit}', fontsize=8)
    axes[0, digit].axis('off')
    idx2 = np.where(y_dig == digit)[0][5]
    axes[1, digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, digit].axis('off')

plt.suptitle('Digits Dataset: Sample Images (two per class)', fontsize=12,
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Define and train models ────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dig, y_dig, test_size=0.2, random_state=42, stratify=y_dig
)

dig_models = {
    'KNN (k=5)
(CONTROL)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Logistic Reg
(Challenger 1)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'Random Forest
(Challenger 2)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    'SVM (RBF)
(Challenger 3)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
}

dig_trained = {}
print(f"{'Model':<35} {'Accuracy':>10} {'F1 (macro)':>12}")
print("-" * 60)
for name, model in dig_models.items():
    model.fit(X_tr, y_tr)
    dig_trained[name] = model
    acc = accuracy_score(y_te, model.predict(X_te))
    f1  = f1_score(y_te, model.predict(X_te), average='macro')
    print(f"  {name.replace(chr(10),' '):<33} {acc:>10.4f} {f1:>12.4f}")

In [ ]:
# ── Multi-class A/B test using bootstrap ──────────────────────
dig_control_name = 'KNN (k=5)
(CONTROL)'
dig_control = dig_trained[dig_control_name]

print("=" * 65)
print("  BOOTSTRAP A/B TEST: DIGITS (F1-macro as metric)")
print("=" * 65)

dig_boot_results = {}
for name, model in dig_trained.items():
    if name == dig_control_name: continue

    result = compare_models_bootstrap(
        dig_control, model, X_te, y_te,
        metric_fn=lambda y, yp: f1_score(y, yp, average='macro'),
        n_bootstrap=1000
    )
    dig_boot_results[name] = result

    print(f"
  Challenger: {name.replace(chr(10),' ')}")
    print(f"    Control F1   : {result['score_A']:.4f}  95% CI: ({result['ci_A'][0]:.4f}, {result['ci_A'][1]:.4f})")
    print(f"    Challenger F1: {result['score_B']:.4f}  95% CI: ({result['ci_B'][0]:.4f}, {result['ci_B'][1]:.4f})")
    print(f"    Diff (B-A)   : {result['mean_diff']:+.4f}  95% CI: ({result['ci_diff'][0]:+.4f}, {result['ci_diff'][1]:+.4f})")
    print(f"    p-value      : {result['p_value']:.4f}  -> {'** SIGNIFICANT **' if result['significant'] else 'not significant'}")

In [ ]:
# ── Visualise: digits model comparison ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_dig = [GREY, BLUE, GREEN, ORANGE]
dig_names_short = ['KNN (Control)', 'Logistic Reg', 'Random Forest', 'SVM (RBF)']

# 1. Bootstrap distributions
ax = axes[0, 0]
for (name, result), color in zip(dig_boot_results.items(), [BLUE, GREEN, ORANGE]):
    ax.hist(result['diffs']*100, bins=40, color=color, alpha=0.5, density=True,
            edgecolor='none', label=name.split('
')[0])
ax.axvline(0, color='black', linewidth=2, linestyle='--', label='No difference')
ax.set_xlabel('F1 difference (pp)')
ax.set_title('Bootstrap: Challenger - Control (F1-macro)')
ax.legend(fontsize=8)

# 2. Per-class accuracy heatmap (confusion matrix for best model)
best_name = max(dig_boot_results, key=lambda n: dig_boot_results[n]['score_B'])
best_model = dig_trained[best_name]
cm = confusion_matrix(y_te, best_model.predict(X_te))
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ax2 = axes[0, 1]
sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='Blues', ax=ax2,
            xticklabels=range(10), yticklabels=range(10))
ax2.set_title(f'Confusion Matrix: {best_name.split(chr(10))[0]}')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')

# 3. CV box plots
ax3 = axes[1, 0]
cv_data = []
for name, model in dig_trained.items():
    scores = cross_val_score(model, X_dig, y_dig, cv=10, scoring='accuracy')
    cv_data.append(scores)

bp = ax3.boxplot(cv_data, patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], colors_dig):
    patch.set_facecolor(color); patch.set_alpha(0.55)
ax3.set_xticklabels(dig_names_short, fontsize=8)
ax3.set_ylabel('10-fold CV Accuracy')
ax3.set_title('CV Score Distributions (notched = 95% CI of median)')

# 4. Summary: 95% CI comparison
ax4 = axes[1, 1]
means = [np.mean(d) for d in cv_data]
sems  = [np.std(d) / np.sqrt(10) for d in cv_data]
cis   = [1.96 * s for s in sems]

for i, (name, mean, ci, color) in enumerate(
        zip(dig_names_short, means, cis, colors_dig)):
    ax4.barh(i, mean, color=color, alpha=0.65, edgecolor='none')
    ax4.errorbar(mean, i, xerr=ci, fmt='none', color='black', capsize=5, linewidth=2)
    ax4.text(mean + 0.002, i, f'{mean:.4f}', va='center', fontsize=9)

ax4.set_yticks(range(len(dig_names_short)))
ax4.set_yticklabels(dig_names_short, fontsize=9)
ax4.set_xlabel('Mean CV Accuracy')
ax4.set_title('Point Estimates +- 95% CI')
ax4.set_xlim(min(means)*0.985, 1.005)

plt.suptitle('Digits Dataset: Model A/B Comparison', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## Part 8 — Bayesian A/B Testing

**Why Bayesian?**
- Frequentist A/B gives you: "Assuming H0, how likely is this data?"
- Bayesian A/B gives you: "Given this data, what's the probability B is better than A?"
- No peeking problem — you can update beliefs continuously
- Gives you **probability of being the winner**, not just significance

### The Math (for conversion rates)

Prior: Beta(alpha, beta) — encodes prior beliefs about conversion rate  
Likelihood: Binomial (conversions out of trials)  
Posterior: Beta(alpha + conversions, beta + non-conversions) — conjugate prior!

$$P(B > A | data) = P(\theta_B > \theta_A)$$

Estimated by Monte Carlo sampling from both posteriors.


In [ ]:
# ── Bayesian A/B test: Beta-Binomial conjugate model ──────────

def bayesian_ab_test(conv_A, n_A, conv_B, n_B,
                     prior_alpha=1, prior_beta=1,
                     n_samples=100000):
    # Bayesian A/B test: Beta-Binomial conjugate. Returns P(B>A), uplift, CIs.
    # Posterior distributions
    post_A = beta_dist(prior_alpha + conv_A, prior_beta + n_A - conv_A)
    post_B = beta_dist(prior_alpha + conv_B, prior_beta + n_B - conv_B)

    # Monte Carlo samples
    samples_A = post_A.rvs(n_samples)
    samples_B = post_B.rvs(n_samples)

    prob_B_beats_A = np.mean(samples_B > samples_A)
    expected_uplift = (samples_B - samples_A).mean()
    uplift_ci = (np.percentile(samples_B - samples_A, 2.5),
                 np.percentile(samples_B - samples_A, 97.5))

    return {
        'prob_B_beats_A': prob_B_beats_A,
        'expected_uplift': expected_uplift,
        'uplift_ci_95': uplift_ci,
        'post_A': post_A, 'post_B': post_B,
        'samples_A': samples_A, 'samples_B': samples_B,
        'mean_A': post_A.mean(), 'mean_B': post_B.mean(),
        'ci_A': post_A.ppf([0.025, 0.975]),
        'ci_B': post_B.ppf([0.025, 0.975]),
    }

# ── Example: checkout redesign ────────────────────────────────
conv_A, n_A = group_A.sum(), n_A
conv_B, n_B = group_B.sum(), n_B

result = bayesian_ab_test(conv_A, n_A, conv_B, n_B)

print("=" * 55)
print("  BAYESIAN A/B TEST: CHECKOUT REDESIGN")
print("=" * 55)
print(f"  Data: A={conv_A}/{n_A}, B={conv_B}/{n_B}")
print()
print(f"  Posterior mean A : {result['mean_A']*100:.2f}%")
print(f"  95% Credible Int : ({result['ci_A'][0]*100:.2f}%, {result['ci_A'][1]*100:.2f}%)")
print()
print(f"  Posterior mean B : {result['mean_B']*100:.2f}%")
print(f"  95% Credible Int : ({result['ci_B'][0]*100:.2f}%, {result['ci_B'][1]*100:.2f}%)")
print()
print(f"  P(B > A)         : {result['prob_B_beats_A']*100:.1f}%")
print(f"  Expected uplift  : {result['expected_uplift']*100:+.2f}pp")
print(f"  95% CI for uplift: ({result['uplift_ci_95'][0]*100:+.2f}pp, {result['uplift_ci_95'][1]*100:+.2f}pp)")
print()
if result['prob_B_beats_A'] > 0.95:
    print("  DEPLOY B: >95% probability B is better")
elif result['prob_B_beats_A'] > 0.80:
    print("  LEAN B: collect more data for confidence")
else:
    print("  INCONCLUSIVE: keep running or keep A")

In [ ]:
# ── Bayesian updating over time ────────────────────────────────
np.random.seed(42)
true_pA, true_pB = 0.10, 0.13
days_bayes = 30; daily = 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

prob_over_time = []
uplift_over_time = []
cum_cA, cum_nA, cum_cB, cum_nB = 0, 0, 0, 0

snapshots = [5, 15, 30]
snapshot_results = {}

for day in range(1, days_bayes+1):
    cum_cA += np.random.binomial(daily, true_pA)
    cum_nA += daily
    cum_cB += np.random.binomial(daily, true_pB)
    cum_nB += daily
    r = bayesian_ab_test(cum_cA, cum_nA, cum_cB, cum_nB, n_samples=20000)
    prob_over_time.append(r['prob_B_beats_A'])
    uplift_over_time.append(r['expected_uplift'] * 100)
    if day in snapshots:
        snapshot_results[day] = r

# 1. P(B>A) over time
axes[0,0].plot(range(1, days_bayes+1), [p*100 for p in prob_over_time],
               color=PURPLE, linewidth=2.5)
axes[0,0].axhline(95, color=RED, linestyle='--', linewidth=1.5, label='95% threshold')
axes[0,0].axhline(80, color=ORANGE, linestyle=':', linewidth=1.5, label='80% threshold')
axes[0,0].set_xlabel('Day'); axes[0,0].set_ylabel('P(B > A) %')
axes[0,0].set_title('Bayesian Belief: P(B better than A) over time')
axes[0,0].legend(fontsize=9); axes[0,0].set_ylim(0, 100)

# 2. Posterior distributions at different points
x_range = np.linspace(0.04, 0.22, 300)
ax2 = axes[0,1]
styles = ['-', '--', ':']
for (day, r), ls in zip(snapshot_results.items(), styles):
    post_A = beta_dist(1 + cum_cA, 1 + cum_nA - cum_cA)
    post_B = beta_dist(1 + cum_cB, 1 + cum_nB - cum_cB)
    ax2.plot(x_range, r['post_A'].pdf(x_range), color=BLUE, linestyle=ls,
             linewidth=1.8, label=f'A (day {day})')
    ax2.plot(x_range, r['post_B'].pdf(x_range), color=GREEN, linestyle=ls,
             linewidth=1.8, label=f'B (day {day})')
ax2.set_xlabel('Conversion Rate')
ax2.set_ylabel('Posterior Density')
ax2.set_title('Posterior Distributions at Different Days')
ax2.legend(fontsize=7, ncol=2)

# 3. Expected uplift over time
axes[1,0].plot(range(1, days_bayes+1), uplift_over_time, color=GREEN, linewidth=2.5)
axes[1,0].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1,0].axhline((true_pB-true_pA)*100, color=RED, linestyle=':', linewidth=1.5,
                  label=f'True uplift={(true_pB-true_pA)*100:.0f}pp')
axes[1,0].set_xlabel('Day'); axes[1,0].set_ylabel('Expected Uplift (pp)')
axes[1,0].set_title('Expected Uplift Estimate Over Time')
axes[1,0].legend(fontsize=9)

# 4. Final posterior comparison
final = snapshot_results[30]
ax4 = axes[1,1]
x_fine = np.linspace(0.04, 0.22, 400)
ax4.plot(x_fine, final['post_A'].pdf(x_fine), color=BLUE, linewidth=2.5, label='Posterior A')
ax4.plot(x_fine, final['post_B'].pdf(x_fine), color=GREEN, linewidth=2.5, label='Posterior B')
ax4.fill_between(x_fine, final['post_A'].pdf(x_fine), alpha=0.2, color=BLUE)
ax4.fill_between(x_fine, final['post_B'].pdf(x_fine), alpha=0.2, color=GREEN)
ax4.axvline(final['post_A'].mean(), color=BLUE, linestyle='--', linewidth=1.8)
ax4.axvline(final['post_B'].mean(), color=GREEN, linestyle='--', linewidth=1.8)
ax4.set_xlabel('Conversion Rate')
ax4.set_ylabel('Density')
ax4.set_title(f'Final Posteriors (Day 30)
P(B>A)={prob_over_time[-1]*100:.1f}%')
ax4.legend(fontsize=9)

plt.suptitle('Bayesian A/B Testing — Continuous Belief Updating', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## Part 9 — Multi-Armed Bandit: When Classic A/B Is Too Slow

**The problem with classic A/B:** You split traffic 50/50 for the whole experiment, even while you're gathering evidence that one arm is clearly better. That costs you conversions during the experiment.

**Multi-Armed Bandit (MAB):** Adaptively allocates more traffic to better-performing variants while still exploring.

### Three Strategies
- **Epsilon-Greedy:** With probability epsilon, explore randomly; otherwise exploit the best known arm
- **UCB1 (Upper Confidence Bound):** Explore arms whose uncertainty is high; exploit when certain
- **Thompson Sampling:** Sample from posterior distributions; naturally balances explore/exploit


In [ ]:
# ── Simulate: 3 variants, Thompson Sampling vs classic A/B ────
np.random.seed(123)

true_rates = [0.10, 0.13, 0.115]   # Arm 0=control, 1=best, 2=medium
n_rounds = 5000
n_arms = len(true_rates)
variant_names = ['Control (10%)', 'Variant B (13%)', 'Variant C (11.5%)']

# ── Thompson Sampling ──────────────────────────────────────────
alpha_ts = np.ones(n_arms)   # Beta prior alpha
beta_ts  = np.ones(n_arms)   # Beta prior beta
ts_choices = []; ts_rewards = []

for t in range(n_rounds):
    # Sample from each arm's posterior
    samples = [np.random.beta(a, b) for a, b in zip(alpha_ts, beta_ts)]
    chosen = np.argmax(samples)
    reward = np.random.binomial(1, true_rates[chosen])
    alpha_ts[chosen] += reward
    beta_ts[chosen]  += (1 - reward)
    ts_choices.append(chosen); ts_rewards.append(reward)

# ── Classic A/B (equal split) ──────────────────────────────────
ab_choices = []; ab_rewards = []
for t in range(n_rounds):
    chosen = t % n_arms   # Round-robin
    reward = np.random.binomial(1, true_rates[chosen])
    ab_choices.append(chosen); ab_rewards.append(reward)

# ── Compute cumulative regret ──────────────────────────────────
best_rate = max(true_rates)

def cumulative_regret(choices, rewards):
    regret = [best_rate - true_rates[c] for c in choices]
    return np.cumsum(regret)

ts_regret = cumulative_regret(ts_choices, ts_rewards)
ab_regret = cumulative_regret(ab_choices, ab_rewards)

print("Results after", n_rounds, "rounds:")
print()
print(f"{'Strategy':<22} {'Total Reward':>14} {'Cumulative Regret':>18} {'Best Arm %':>12}")
print("-" * 70)
for choices, rewards, name in [
        (ts_choices, ts_rewards, 'Thompson Sampling'),
        (ab_choices, ab_rewards, 'Classic A/B (equal)')]:
    total_r = sum(rewards)
    final_regret = cumulative_regret(choices, rewards)[-1]
    best_arm_pct = sum(c == np.argmax(true_rates) for c in choices) / n_rounds * 100
    print(f"  {name:<20} {total_r:>14} {final_regret:>18.1f} {best_arm_pct:>11.1f}%")

print()
print(f"Thompson Sampling gained {sum(ts_rewards) - sum(ab_rewards)} extra conversions!")

In [ ]:
# ── Visualise MAB ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

rounds = range(1, n_rounds+1)

# 1. Cumulative regret
axes[0,0].plot(rounds, ts_regret, color=GREEN, linewidth=2.5, label='Thompson Sampling')
axes[0,0].plot(rounds, ab_regret, color=RED,   linewidth=2.5, label='Classic A/B')
axes[0,0].set_xlabel('Round'); axes[0,0].set_ylabel('Cumulative Regret')
axes[0,0].set_title('Cumulative Regret: Thompson vs Classic A/B')
axes[0,0].legend(fontsize=10)

# 2. Arm selection frequency over time
window = 500
ts_arm_freq = pd.DataFrame(
    [[1 if c==i else 0 for i in range(n_arms)] for c in ts_choices],
    columns=variant_names
).rolling(window).mean()

for col, color in zip(variant_names, [GREY, BLUE, ORANGE]):
    axes[0,1].plot(rounds[window:], ts_arm_freq[col].iloc[window:]*100,
                   color=color, linewidth=2, label=col)
axes[0,1].set_xlabel('Round'); axes[0,1].set_ylabel('% Traffic to Arm')
axes[0,1].set_title(f'Thompson Sampling: Traffic Allocation (rolling {window})')
axes[0,1].legend(fontsize=8); axes[0,1].set_ylim(0, 100)

# 3. Cumulative conversion rate
ts_cum_rate  = np.cumsum(ts_rewards)  / np.arange(1, n_rounds+1) * 100
ab_cum_rate  = np.cumsum(ab_rewards)  / np.arange(1, n_rounds+1) * 100
axes[1,0].plot(rounds, ts_cum_rate, color=GREEN, linewidth=2.5, label='Thompson Sampling')
axes[1,0].plot(rounds, ab_cum_rate, color=RED,   linewidth=2.5, label='Classic A/B')
axes[1,0].axhline(best_rate*100, color='black', linestyle='--', linewidth=1.5,
                  label=f'Optimal rate ({best_rate*100}%)')
axes[1,0].set_xlabel('Round'); axes[1,0].set_ylabel('Conversion Rate (%)')
axes[1,0].set_title('Cumulative Conversion Rate Over Time')
axes[1,0].legend(fontsize=9)

# 4. Final posterior beliefs (Thompson)
x_r = np.linspace(0.0, 0.25, 300)
colors_arms = [GREY, BLUE, ORANGE]
for i, (name, color) in enumerate(zip(variant_names, colors_arms)):
    posterior = beta_dist(alpha_ts[i], beta_ts[i])
    axes[1,1].plot(x_r, posterior.pdf(x_r), color=color, linewidth=2.5, label=name)
    axes[1,1].axvline(true_rates[i], color=color, linestyle=':', linewidth=1.5)
axes[1,1].set_xlabel('Conversion Rate')
axes[1,1].set_title("Final Posterior Beliefs
(dotted = true rates)")
axes[1,1].legend(fontsize=8)

plt.suptitle('Multi-Armed Bandit: Thompson Sampling vs Classic A/B', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## Part 10 — Final Decision Framework

A complete checklist for every A/B test or model comparison you run.


In [ ]:
# ── Decision framework printer ─────────────────────────────────
lines = [
    "+================================================================+",
    "|           A/B TEST & MODEL SELECTION DECISION FRAMEWORK        |",
    "+================================================================+",
    "|  BEFORE STARTING                                               |",
    "|  [ ] Define primary metric (ONE metric drives the decision)    |",
    "|  [ ] Define secondary metrics (for guardrails)                 |",
    "|  [ ] Set alpha=0.05 and power=0.80 BEFORE looking at data      |",
    "|  [ ] Calculate required sample size from MDE                   |",
    "|  [ ] Set experiment duration and stick to it                   |",
    "|  [ ] Define what success looks like in advance                 |",
    "|  DURING EXPERIMENT                                             |",
    "|  [ ] Do NOT peek and adjust based on interim results           |",
    "|  [ ] Monitor for data quality issues (null rates, outliers)    |",
    "|  [ ] Check for sample ratio mismatch (SRM)                     |",
    "|  [ ] Log everything for post-hoc analysis                      |",
    "|  AFTER EXPERIMENT                                              |",
    "|  [ ] Run pre-planned test (z-test / t-test / chi2)             |",
    "|  [ ] Check assumptions (normality, independence, EV>=5)        |",
    "|  [ ] Report effect size + confidence interval, not just p      |",
    "|  [ ] Segment analysis: does effect hold across subgroups?      |",
    "|  [ ] Check for Simpsons paradox                                |",
    "|  [ ] Compute practical significance (is lift worth deploying?) |",
    "|  FOR ML MODEL SELECTION SPECIFICALLY                           |",
    "|  [ ] Use stratified splits (same class balance in all folds)   |",
    "|  [ ] Compare distributions of scores, not just point estimates |",
    "|  [ ] Use McNemar test to compare error patterns                |",
    "|  [ ] Report metrics relevant to business (not just accuracy)   |",
    "|  [ ] Consider: latency, memory, maintainability tradeoffs      |",
    "|  [ ] Shadow-deploy challenger before full rollout              |",
    "|  WHEN TO USE WHAT                                              |",
    "|  Two proportions (CTR, conversion): z-test / chi-square        |",
    "|  Two means (revenue, time): t-test                             |",
    "|  Model accuracy: bootstrap / McNemar / CV t-test               |",
    "|  Many variants: Bonferroni or ANOVA correction                 |",
    "|  Want P(B>A): Bayesian Beta-Binomial                           |",
    "|  High traffic, low regret tolerance: Thompson Sampling         |",
    "+================================================================+",
]
for line in lines:
    print(line)

In [ ]:
# ── Quick-start template ───────────────────────────────────────
template_lines = [
    "# A/B TEST TEMPLATE",
    "from scipy.stats import norm",
    "import numpy as np",
    "",
    "# 1. PRE-REGISTRATION",
    "alpha = 0.05; power = 0.80",
    "baseline_rate = 0.10",
    "mde_absolute  = 0.02   # minimum lift that matters",
    "",
    "# 2. SAMPLE SIZE",
    "def required_n(p1, mde, alpha=0.05, power=0.80):",
    "    p2 = p1 + mde; p_bar = (p1+p2)/2",
    "    za = norm.ppf(1-alpha/2); zb = norm.ppf(power)",
    "    return int(np.ceil((za*np.sqrt(2*p_bar*(1-p_bar))+zb*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2/mde**2))",
    "",
    "n_per_group = required_n(baseline_rate, mde_absolute)",
    "print(f'Required n per group: {n_per_group:,}')",
    "",
    "# 3. COLLECT DATA",
    "# conv_A, n_A = [conversions], [total]",
    "# conv_B, n_B = [conversions], [total]",
    "",
    "# 4. RUN TEST",
    "# p_pool = (conv_A + conv_B) / (n_A + n_B)",
    "# p_A = conv_A/n_A; p_B = conv_B/n_B",
    "# se = np.sqrt(p_pool*(1-p_pool)*(1/n_A+1/n_B))",
    "# z  = (p_B-p_A)/se",
    "# p_val = 2*(1-norm.cdf(abs(z)))",
    "",
    "# 5. REPORT",
    "# lift = (p_B-p_A)*100",
    "# decision = 'DEPLOY B' if p_val<alpha and p_B>p_A else 'KEEP A'",
]
print("
".join(template_lines))

---
## Summary & Key Takeaways

| Concept | One-line |
|---------|---------|
| **Null hypothesis** | Assume no difference. Reject only with overwhelming evidence. |
| **p-value** | P(data this extreme | H0 true). NOT P(H0 true). |
| **Alpha** | Your false positive budget. Set it before looking at data. |
| **Power** | Probability of detecting a real effect. Needs large n. |
| **MDE** | Smallest lift you care about. Drives sample size. |
| **Peeking** | Looking at p-value before planned end inflates false positives. |
| **Effect size** | Always report. Significance without effect size is useless. |
| **Bootstrap** | Resample test set; get uncertainty on any model metric. |
| **McNemar** | Tests if two models fail on the same samples (paired). |
| **Bayesian** | P(B > A | data). No peeking problem. Needs prior. |
| **Thompson Sampling** | Adaptive traffic allocation. Fewer wasted impressions. |

---
*Notebook: A/B Testing — From First Principles to ML Model Selection*
*Dependencies: numpy, pandas, scipy, scikit-learn, matplotlib, seaborn*
